# Optimal Execution Visual Lab

## Executive summary
Execution speed exchanges market impact for timing risk. This notebook uses one reproducible synthetic market to connect Almgren–Chriss scheduling, resilient liquidity, a reactive limit-order book, transaction-cost analysis, and bounded reinforcement-learning adjustments. Economic implementation shortfall is kept separate from the RL training reward.

### 日本語サマリー
執行を速めると価格変動リスクは下がりますが、市場インパクトは上がります。本ノートブックは、Almgren–Chriss、板の回復力、反応型板、取引コスト分析、制約付き強化学習を、同一の再現可能な合成市場で比較します。RLの整形報酬と経済的な実装ショートフォールは明確に分離します。

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from optimal_execution.config import load_config
from optimal_execution.provenance import artifact_dirs

config_path = Path(os.environ.get("OPTIMAL_EXECUTION_CONFIG", "configs/quick.yaml"))
cfg = load_config(config_path)
paths = artifact_dirs(cfg)
display(
    Markdown(
        f"**Profile:** `{cfg.profile}` · **seed:** `{cfg.seed}` · **order:** `{cfg.initial_inventory:,.0f}` shares · **horizon:** `{cfg.horizon_seconds:.0f}` seconds"
    )
)

## Concept map and market-microstructure definitions

**Spread** is the immediate cost of crossing the best quotes. **Liquidity** combines displayed depth and replenishment. **Market impact** is the agent's own price footprint. **Timing risk** is the uncertainty from holding inventory while the unaffected price moves. An **execution policy** maps time, inventory, and book state into bounded market and limit orders.

Strategic baseline (AC / resilient schedule) → tactical heuristic or RL adjustment → safety layer (inventory, participation, price collar, deadline).

In [ ]:
display(
    pd.DataFrame(
        {
            "variable": [
                "side",
                "arrival price",
                "initial inventory",
                "steps",
                "annualized volatility",
                "ADV",
                "spread",
                "participation cap",
            ],
            "value": [
                cfg.side,
                cfg.arrival_price,
                cfg.initial_inventory,
                cfg.n_decision_steps,
                cfg.annualized_volatility,
                cfg.average_daily_volume,
                cfg.spread_bps,
                cfg.max_participation_rate,
            ],
            "unit": [
                "",
                "price/share",
                "shares",
                "",
                "fraction/year",
                "shares/day",
                "bps",
                "fraction",
            ],
        }
    )
)

## Synthetic market profiles and unaffected price paths
Volume, volatility, spread, and depth follow documented intraday profiles with local stochastic multipliers. The unaffected price contains no agent impact; impacted prices are layered on top. Common random numbers make strategy differences paired rather than independent.

In [ ]:
display(Image(filename=str(paths["figures"] / "unaffected_price_paths.png")))

## Temporary, permanent, transient, and square-root impact

Three linear channels are kept distinct. **Temporary** impact is the price concession paid only while trading, $g(v_k)=\eta\,v_k$ per share (rate $v_k=q_k/\Delta t$), a cash cost over a step of $C^{\text{temp}}_k=\eta\,q_k^2/\Delta t$. It leaves no trace once the trade stops. **Permanent** impact is a lasting mid shift $\gamma\sum_{j<k}q_j$ that never decays and is the footprint later trades inherit. **Transient** impact is a displacement state $D$ that builds with trading and relaxes at resilience rate $\rho$, using a point-impulse convention

$$D_{k+1}=e^{-\rho\,\Delta t}\left(D_k+\eta_t q_k\right).$$

The execution at step $k$ pays the pre-trade $D_k$ plus half of its own jump, $D_k+\tfrac{1}{2}\eta_t q_k$ (a block-average through the book). Sign convention: every quantity is a positive adverse magnitude, so a sell fills at $S^0-a$ and a buy at $S^0+a$.

The exponential recursion is the $\rho$-kernel special case of a propagator

$$D_k=\sum_{j<k} G\!\left((k-j)\Delta t\right)\eta_t q_j,\qquad G(0)=1,$$

for exponential or completely-monotone power-law kernels. Because the simulator exposes these latent states, the per-trade split into temporary/permanent/transient is exact inside the model — a luxury real data never grants.

The square-root law $I(Q)=Y\,\sigma_{\text{day}}\sqrt{Q/\mathrm{ADV}}$ is a concave law for the **total** impact of a metaorder of size $Q$. It is shown only as an empirical diagnostic and is deliberately not added on top of the linear channels, which would double-count.

In [ ]:
for name in ["impact_channels.png", "impact_recovery.png", "sqrt_impact.png"]:
    display(Image(filename=str(paths["figures"] / name)))

## Almgren–Chriss derivation, sensitivities, and efficient frontier

This is the continuous-time mean–variance program for selling $X$ shares over $[0,T]$. With linear temporary impact $\eta$ and price volatility $\sigma$, minimise

$$J[x]=\int_0^T\!\left(\eta\,\dot{x}_t^2+\lambda\,\sigma^2 x_t^2\right)dt,\qquad x_0=X,\ x_T=0.$$

The first term is the impact cost of trading fast ($\dot{x}$ is the trading rate); the second prices timing risk — holding inventory $x$ while the price diffuses — scaled by risk aversion $\lambda$.

The Euler–Lagrange condition gives $\ddot{x}=\kappa^2 x$ with urgency $\kappa=\sqrt{\lambda\sigma^2/\eta}$, whose boundary-matched solution is

$$x_t^{*}=X\,\frac{\sinh\!\left(\kappa(T-t)\right)}{\sinh(\kappa T)},\qquad x_t^{*}\to X\!\left(1-\frac{t}{T}\right)\ \text{as}\ \kappa\to0.$$

The $\kappa\to0$ limit is TWAP. Larger $\lambda$ or $\sigma$ (more urgency) front-loads execution; larger $\eta$ (dearer impact) flattens it toward TWAP. Order size $X$ scales the quantities but not the normalised shape.

On the decision grid the child order is $q_k=x_k-x_{k+1}$, with expected shortfall and timing-risk variance

$$E[C]=\frac{\gamma}{2}X^2+\eta\sum_k\frac{q_k^2}{\Delta t},\qquad V[C]=\sum_k \sigma_k^2\,\Delta t\,x_{k+1}^2$$

(expected cost also carries a half-spread$\,\cdot X$ and fees). Sweeping $\lambda$ traces the **efficient frontier** of expected cost versus timing-risk standard deviation — the menu of optimal speed/risk trade-offs.

In [ ]:
display(Image(filename=str(paths["figures"] / "ac_trajectories.png")))
display(Image(filename=str(paths["figures"] / "efficient_frontier.png")))

## Obizhaeva–Wang-style resilient liquidity

Where AC penalises speed through a temporary-impact rate, OW models a **resilient** book: trading pushes a purely transient displacement $D$ that decays at rate $\rho$,

$$dD_t=-\rho\,D_t\,dt+\eta_t\,v_t\,dt,\qquad D_{k+1}=e^{-\rho\,\Delta t}\left(D_k+\eta_t q_k\right).$$

Executed against a block-shaped book, the expected cost of a schedule is the quadratic form

$$C(q)=\eta_t\!\left[\tfrac{1}{2}\sum_k q_k^2+\sum_{j<k} q_k q_j\,G\!\left((k-j)\Delta t\right)\right]=\tfrac{1}{2}\eta_t\,q^{\top} M q,\qquad M_{ij}=G\!\left(|i-j|\,\Delta t\right),\ M_{ii}=1.$$

$M$ is symmetric positive definite for exponential (and completely-monotone power-law) kernels, so the scheduling QP is well posed.

Two solvers agree. The classical risk-neutral Obizhaeva–Wang (2013) closed form places equal block trades of $X/(\rho T+2)$ at $t=0$ and $t=T$ and trades at a constant interior rate $\rho X/(\rho T+2)$. The discrete solver minimises $\tfrac{1}{2}\eta_t\,q^{\top}Mq$ under $\sum_k q_k=X,\ q\ge0$ via

$$q=\frac{M^{-1}\mathbf{1}}{\mathbf{1}^{\top} M^{-1}\mathbf{1}}\,X$$

with a small active-set loop, valid for any kernel. Low resilience (small $\rho$) leaves persistent displacement and rewards patience; high $\rho$ lets liquidity replenish and supports faster reuse.

This lab implements the exact risk-neutral continuous solution and the discrete convex optimiser only. The **risk-averse** extension of OW is not implemented — risk preferences enter this suite solely through the Almgren–Chriss layer (see METHODOLOGY.md).

In [ ]:
resilience = pd.read_csv(paths["data"] / "resilience_sweep.csv")
display(resilience.groupby(["rho", "method"], as_index=False)[["cost", "twap_cost"]].first())

## Reactive limit-order book, queue imbalance, and event state

The simplified book carries one aggregated level-1 quote per side plus a deeper-liquidity density, and tracks spread, queue imbalance, recent flow, volatility, and the agent's own impact. Queue imbalance is

$$I=\frac{Q^{b}-Q^{a}}{Q^{b}+Q^{a}}\in[-1,1].$$

The **unaffected** mid $S^0$ never responds to the agent; the **impacted** fair mid is $\text{mid}=S^0-s\left(\gamma\,\Sigma_{\text{mkt}}+D\right)$ with program sign $s$ — the same permanent-$\gamma$ and transient-$D$ channels as the classical world (passive limit fills add none, since they supply liquidity). Temporary impact here is endogenous: a large agent market order consumes L1 depth, walks deeper liquidity, and widens the spread, rather than paying a separate $\eta$.

A latent short-horizon alpha follows an AR(1), $\alpha_k=\phi\,\alpha_{k-1}+\sigma_\alpha Z_k$, and drives both order flow and the next mid move — the stylised adverse-selection channel. Exogenous market-order volume per sub-step is a Poisson count times a mean size, with a capped log-linear intensity

$$\lambda=\lambda_{\text{base}}\,\exp\!\left(\min(\text{tilt},\,10)\right),\qquad \text{tilt}=b_0+b_1 I+b_2 r+b_3\,\text{stress}+b_4\,\alpha.$$

Pre-drawn uniforms give common random numbers across strategies, so differences are paired; state-dependence enters only through the intensity.

The replay control (`reactive=False`) reuses the same exogenous draws but removes the agent's trace entirely — depth is not consumed, quotes and the mid do not move, and impact states stay untouched. The gap between reactive and replay isolates the agent's own footprint. This is a transparent reactive sandbox, not historical replay or exchange-grade realism.

In [ ]:
display(Image(filename=str(paths["figures"] / "lob_depth.png")))
display(Image(filename=str(paths["figures"] / "queue_imbalance.png")))

## Market-order walk, limit-order queue, fill probability, and adverse selection
Market orders buy immediacy but pay spread and book-walk costs. Passive orders can capture spread, but queue ahead lowers fill probability and deadline cleanup can dominate. Latent short-horizon alpha jointly tilts order flow and the next price move, creating stylized adverse selection.

In [ ]:
display(Image(filename=str(paths["figures"] / "market_vs_limit.png")))
lob_summary = pd.read_csv(paths["metrics"] / "lob_strategy_summary.csv")
display(
    lob_summary[["strategy_id", "is_mean_bps", "cvar95_bps", "fill_rate", "cleanup_qty"]].round(3)
)

## Static schedules, implementation-shortfall distributions, TCA decomposition, and tail risk
Immediate, TWAP, VWAP-style, POV, Almgren–Chriss, and resilience-aware schedules share identical stochastic paths. Positive implementation shortfall is adverse for both sells and buys. Latent timing, spread, temporary, permanent, transient, fee, and cleanup components are exact only inside this synthetic model.

In [ ]:
for name in ["strategy_schedules.png", "is_distributions.png", "tca_decomposition.png"]:
    display(Image(filename=str(paths["figures"] / name)))
classical = pd.read_csv(paths["metrics"] / "classical_strategy_summary.csv")
display(
    classical[
        ["strategy_id", "is_mean_bps", "is_mean_ci_lo_bps", "is_mean_ci_hi_bps", "cvar95_bps"]
    ].round(3)
)

## Reactive versus non-reactive simulation
The control run reuses the same exogenous draws but removes the agent's footprint. The resulting cost gap quantifies what this toy replay understates for the configured order size; it is not an estimate for a real venue.

In [ ]:
display(Image(filename=str(paths["figures"] / "reactive_vs_replay.png")))
display(
    pd.read_csv(paths["metrics"] / "reactive_comparison.csv")[
        ["mode", "is_mean_bps", "cvar95_bps"]
    ].round(3)
)

## RL state, actions, reward, safety layer, and training curves

One episode is one parent order; the agent makes $N$ decisions while the book evolves between them. The 12 normalised observations are time, inventory, spread, both depths, imbalance, recent return and flow, volatility, transient impact, volume, and outstanding limit quantity. The discrete action space is $15=5\times3$: a bounded market multiplier $m\in\{0,0.5,1,1.5,2\}$ times a baseline slice, crossed with a limit directive $l\in\{\text{none},\text{join},\text{improve}\}$.

**Residual** RL uses the Almgren–Chriss schedule as the baseline, so $m=1$ reproduces the classical policy exactly and the network learns bounded deviations around it; **free** RL uses a TWAP-sized baseline. A small shared-body actor–critic MLP is trained with PPO's clipped surrogate

$$L=\mathbb{E}\!\left[\min\!\left(\varrho_t\,\hat{A}_t,\ \operatorname{clip}(\varrho_t,1-\epsilon,1+\epsilon)\,\hat{A}_t\right)\right],\qquad \varrho_t=\frac{\pi_\theta(a_t\mid s_t)}{\pi_{\theta_{\text{old}}}(a_t\mid s_t)},$$

with advantages from GAE, $\hat{A}_t=\sum_{l\ge0}(\gamma\lambda)^l\,\delta_{t+l}$ and $\delta_t=R_t+\gamma V(s_{t+1})-V(s_t)$. The training reward $R_t$ is **shaped** (incremental cost plus inventory/impact penalties); validation and every reported comparison use **economic** implementation shortfall on disjoint held-out seeds — the shaped reward is never presented as profit and loss.

A hard safety layer stays authoritative for scripted and learned policies alike: it forbids over-execution, caps child size and participation against an EMA of realised volume, applies a price collar to market orders, and forces terminal liquidation at the deadline (exempt from the caps, and punitive because it walks the book). Violations are counted and penalised, so gaming the constraints is never rewarded.

In [ ]:
display(Image(filename=str(paths["figures"] / "rl_training_history.png")))
history = pd.read_csv(paths["metrics"] / "rl_training_history.csv")
display(
    history.groupby("run_id", as_index=False)
    .tail(1)[["run_id", "episodes", "train_is_ma_bps", "best_val_is_bps"]]
    .round(3)
)

## In-distribution comparison and stress tests
TWAP, Almgren–Chriss, the mixed heuristic, residual RL, and free RL are tested on fixed held-out seeds. Stress regimes change volatility, liquidity, volume profile, adverse alpha, and spread/resilience. The quick profile has one RL seed, so it cannot establish seed-robust superiority.

In [ ]:
display(Image(filename=str(paths["figures"] / "stress_test_heatmap.png")))
stress = pd.read_csv(paths["metrics"] / "stress_summary.csv")
display(stress.pivot(index="strategy_id", columns="regime", values="is_mean_bps").round(3))

## Feature ablation, RL action diagnostics, and distribution shift
Each residual policy is retrained after masking one observation group. The model-misspecification test keeps the trained policy fixed while changing resilience, depth, spread stress, and liquidity. Differences are simulator evidence, not causal feature importance in real markets.

In [ ]:
display(Image(filename=str(paths["figures"] / "feature_ablation.png")))
display(Image(filename=str(paths["figures"] / "model_misspecification.png")))
display(
    pd.read_csv(paths["metrics"] / "ablation_summary.csv")[
        ["strategy_id", "feature_removed", "is_mean_bps", "delta_vs_full_bps"]
    ].round(3)
)

## What the experiment establishes
It establishes that the implementation is reproducible under recorded seeds; classical urgency responds correctly to risk and impact parameters; the agent changes its synthetic market; limit orders exchange spread for fill risk; and policy rankings can change under stress and model shift.

In [ ]:
best_classical = classical.loc[classical.is_mean_bps.idxmin()]
best_lob = lob_summary.loc[lob_summary.is_mean_bps.idxmin()]
id_test = stress[stress.regime == "in_distribution"]
best_id = id_test.loc[id_test.is_mean_bps.idxmin()]
display(
    Markdown(f"""**Generated quick-run findings**  
- Lowest classical mean cost: `{best_classical.strategy_id}` at **{best_classical.is_mean_bps:.3f} bps**.  
- Lowest reactive-LOB mean cost: `{best_lob.strategy_id}` at **{best_lob.is_mean_bps:.3f} bps**.  
- Lowest in-distribution comparison cost: `{best_id.strategy_id}` at **{best_id.is_mean_bps:.3f} bps**.  
These rankings are conditional on the synthetic calibration.""")
)

## What it does not establish; limitations and next steps
This lab does not establish real-market profitability, causal feature value, exchange realism, or statistical RL superiority. It omits hidden liquidity, latency, multi-venue routing, strategic counterparties, manipulation constraints, cross-impact, and empirical calibration. Next steps are real-data calibration with strict holdouts, multi-asset cross-impact, dealer/RFQ markets, stochastic resilience, safer constrained RL, distributionally robust evaluation, and interaction with rough volatility or Hawkes flow.

## References

Almgren–Chriss, Obizhaeva–Wang (2013), and PPO are cited directly in the code; the remaining entries are the canonical sources for the implemented ideas and for the roadmap items in the limitations section.

**Core strategy models**
- R. Almgren & N. Chriss (2001), "Optimal Execution of Portfolio Transactions," *Journal of Risk* 3(2) — the mean–variance execution program, the closed-form `sinh` trajectory, and the efficient frontier implemented in `almgren_chriss.py`.
- D. Bertsimas & A. Lo (1998), "Optimal Control of Execution Costs," *Journal of Financial Markets* 1(1) — the risk-neutral dynamic-programming precursor to Almgren–Chriss.
- A. Obizhaeva & J. Wang (2013), "Optimal Trading Strategy and Supply/Demand Dynamics," *Journal of Financial Markets* 16(1) — resilient supply/demand and the block–constant-rate–block schedule in `resilience.py`.

**Impact and microstructure foundations**
- J.-P. Bouchaud, Y. Gefen, M. Potters & M. Wyart (2004), "Fluctuations and Response in Financial Markets," *Quantitative Finance* 4(2) — the propagator (decay-kernel) view generalising the transient channel.
- J. Gatheral (2010), "No-Dynamic-Arbitrage and Market Impact," *Quantitative Finance* 10(7) — consistency constraints between impact and decay kernels (a roadmap item).
- R. Almgren, C. Thum, E. Hauptmann & H. Li (2005), "Direct Estimation of Equity Market Impact," *Risk* 18(7) — the empirical square-root-law estimation behind the `sqrt_impact` diagnostic.
- B. Tóth et al. (2011), "Anomalous Price Impact and the Critical Nature of Liquidity in Financial Markets," *Physical Review X* 1(2) — universality of the square-root law.
- A. Kyle (1985), "Continuous Auctions and Insider Trading," *Econometrica* 53(6) — linear permanent impact and the informed-flow view behind the latent-alpha adverse-selection channel.
- A. Perold (1988), "The Implementation Shortfall: Paper Versus Real Portfolios," *Journal of Portfolio Management* 14(3) — the cost metric used throughout.

**Reinforcement learning**
- J. Schulman, F. Wolski, P. Dhariwal, A. Radford & O. Klimov (2017), "Proximal Policy Optimization Algorithms," arXiv:1707.06347 — the PPO clipped-surrogate objective in `rl_training.py`.
- J. Schulman, P. Moritz, S. Levine, M. Jordan & P. Abbeel (2016), "High-Dimensional Continuous Control Using Generalized Advantage Estimation," ICLR / arXiv:1506.02438 — GAE.
- Y. Nevmyvaka, Y. Feng & M. Kearns (2006), "Reinforcement Learning for Optimized Trade Execution," *ICML* — the seminal execution-RL study.
- D. Hendricks & D. Wilcox (2014), "A Reinforcement Learning Extension to the Almgren–Chriss Framework for Optimal Trade Execution," *IEEE CIFEr* — RL adjustments around an AC baseline, the residual-RL idea.

**Simulator design and next steps**
- W. Huang, C.-A. Lehalle & M. Rosenbaum (2015), "Simulating and Analyzing Order Book Data: The Queue-Reactive Model," *Journal of the American Statistical Association* 110(509) — the canonical reactive-book reference.
- J. Gatheral, T. Jaisson & M. Rosenbaum (2018), "Volatility is Rough," *Quantitative Finance* 18(6) — rough-volatility regimes (a roadmap item).
- E. Bacry, I. Mastromatteo & J.-F. Muzy (2015), "Hawkes Processes in Finance," *Market Microstructure and Liquidity* 1(1) — Hawkes-style order flow (a roadmap item).

**Textbooks**
- Á. Cartea, S. Jaimungal & J. Penalva (2015), *Algorithmic and High-Frequency Trading*, Cambridge University Press.
- O. Guéant (2016), *The Financial Mathematics of Market Liquidity: From Optimal Execution to Market Making*, CRC Press.